# Laboratorio 7: Spark MLlib

## Inciso 7. Entrenamiento final y evaluación en 2026

**Equipo:** Abby Donis (22440), Hansel Lopez (19026), Fabian Prado (23427)

**Datos.** Se cargan los modelos guardados de los incisos 5 y 6 (entrenados con 2025T1-T3 y validados en 2025T4) y se evalúan sobre 2026T1, que no se usó en el desarrollo.

**Tarea.** Comparar el desempeño de regresión lineal y Random Forest sobre datos completamente no vistos.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from pyspark.ml import PipelineModel

from src import config, datos, modelos

config.preparar_directorios()
pd.set_option("display.float_format", "{:,.3f}".format)

spark = datos.iniciar_spark()
df_2026 = datos.cargar(spark, 2026)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### 7.1 Cargar modelos entrenados

Se cargan los PipelineModel guardados en los incisos anteriores. Estos modelos fueron entrenados con 2025T1-T3 y validados en 2025T4.

In [2]:
modelo_lineal = PipelineModel.load(str(config.MODELOS / "regresion_lineal"))
modelo_rf = PipelineModel.load(str(config.MODELOS / "random_forest"))
print("Modelos cargados correctamente")

Modelos cargados correctamente


### 7.2 Evaluar sobre 2026T1

Se aplican ambos modelos sobre los datos de 2026T1 y se calculan las métricas MAE, RMSE y R2.

In [3]:
print("Registros en 2026T1:", df_2026.count())

Registros en 2026T1: 13258


In [4]:
pred_lineal_2026 = modelo_lineal.transform(df_2026).cache()
pred_rf_2026 = modelo_rf.transform(df_2026).cache()

metricas_lineal_2026 = modelos.metricas(pred_lineal_2026)
metricas_rf_2026 = modelos.metricas(pred_rf_2026)

### 7.3 Comparación con 2025T4

Se comparan las métricas de validación (2025T4) con las de prueba (2026T1) para ver si los modelos generalizan bien.

In [5]:
lineal_json = json.loads((config.MODELOS / "regresion_lineal.json").read_text())
rf_json = json.loads((config.MODELOS / "random_forest.json").read_text())

metricas_lineal_2025 = lineal_json["metricas_validacion"]
metricas_rf_2025 = rf_json["metricas_validacion"]
metricas_ref_2025 = lineal_json["metricas_referencia"]

In [6]:
comparacion = pd.DataFrame({
    "2025T4 (validación)": {
        "referencia": metricas_ref_2025["rmse"],
        "regresión lineal": metricas_lineal_2025["rmse"],
        "random forest": metricas_rf_2025["rmse"],
    },
    "2026T1 (prueba)": {
        "referencia": float("nan"),
        "regresión lineal": metricas_lineal_2026["rmse"],
        "random forest": metricas_rf_2026["rmse"],
    }
})
comparacion

,2025T4 (validación),2026T1 (prueba)
referencia,"2,889.571",NaN
regresión lineal,"2,186.420","2,167.387"
random forest,"1,899.883","1,954.832"


**Lectura.**

- Ambos modelos mantienen un desempeño similar en 2026T1 respecto a 2025T4, lo que indica que generalizan bien.
- Random Forest sigue superando a la regresión lineal en 2026.
- El RMSE en 2026 puede ser ligeramente distinto por diferencias estacionales o cambios en el mercado laboral entre trimestres.

### 7.4 Métricas completas en 2026T1

In [7]:
metricas_completas = pd.DataFrame([metricas_lineal_2026, metricas_rf_2026],
                                  index=["regresión lineal", "random forest"])
metricas_completas

,mae,rmse,r2
regresión lineal,"1,239.175","2,167.387",0.429
random forest,"1,094.173","1,954.832",0.535


**Interpretación.**

- Random Forest mantiene su ventaja sobre la regresión lineal en datos no vistos.
- El R2 en 2026 indica qué tanta variación del salario explican los modelos en un periodo completamente nuevo.
- Si el R2 baja mucho respecto a 2025T4, puede haber cambios estructurales en el mercado laboral que los predictores no capturan.

### 7.5 Conclusión del inciso 7

- Ambos modelos generalizan bien a 2026T1, con métricas similares a las de validación en 2025T4.
- Random Forest sigue siendo superior a la regresión lineal en datos no vistos.
- La evaluación en 2026 confirma que los modelos capturaron patrones reales y no solo ruido del periodo de entrenamiento.

In [8]:
spark.stop()